In [55]:
import json
import pandas as pd
from utilities.extract_qids import extract_qids
print('p1')
from utilities.alignment import (
    align_sections,
    display_alignment,
    display_score_matrix,
    save_alignment_results,
)
print('p2')
from utilities.load_models import load_MPropositioneur_V2, load_Qwen3_8B_via_llama_cpp
print('p3')
from utilities.omission import calculate_omission
print('p4')
from utilities.propositions import extract_propositions
print('p5')
from utilities.triplets import extract_triplets
print('p6')

p1
p2
p3
p4
p5
p6


In [56]:
# download the model propositioner
model_AP, tokenizer_AP = (load_MPropositioneur_V2())

model_triplets = load_Qwen3_8B_via_llama_cpp()
# Extract the two articles

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 519.84it/s]


In [57]:

data_en = extract_qids(
    "en",
    "Falkland_Islands",
    verbose=False,
) 
# data_en and data_es are lists of dictionaries
data_es = extract_qids(
    "es",
    "Islas_Malvinas_(territorio_británico_de_ultramar)",
    verbose=False,
)

with open( "data_es.json", "w", encoding="utf-8" ) as f:
    json.dump(data_es, f, indent=4, ensure_ascii=False)
    print("Triples saved to data_es.json")

with open( "data_en.json", "w", encoding="utf-8" ) as f:
    json.dump(data_en, f, indent=4, ensure_ascii=False)
    print("Triples saved to data_en.json")




   batch 1/10 (25 titles)
   batch 2/10 (25 titles)
   batch 3/10 (25 titles)
   batch 4/10 (25 titles)
   batch 5/10 (25 titles)
   batch 6/10 (25 titles)
   batch 7/10 (25 titles)
   batch 8/10 (25 titles)
   batch 9/10 (25 titles)
   batch 10/10 (4 titles)
   batch 1/10 (25 titles)
   batch 2/10 (25 titles)
   batch 3/10 (25 titles)
   batch 4/10 (25 titles)
   batch 5/10 (25 titles)
   batch 6/10 (25 titles)
   batch 7/10 (25 titles)
   batch 8/10 (25 titles)
   batch 9/10 (25 titles)
   batch 10/10 (4 titles)
   batch 1/5 (25 titles)
   batch 2/5 (25 titles)
   batch 3/5 (25 titles)
   batch 4/5 (25 titles)
   batch 5/5 (25 titles)
   batch 1/5 (25 titles)
   batch 2/5 (25 titles)
   batch 3/5 (25 titles)
   batch 4/5 (25 titles)
   batch 5/5 (25 titles)
Triples saved to data_es.json
Triples saved to data_en.json


In [58]:
# Align their sections

section_pairs, score_matrix = align_sections(
    data_en,
    data_es,
    threshold=0.70,
)

# Display results

display_alignment(section_pairs)
display_score_matrix(score_matrix)





Article A: en:Falkland_Islands
Sections A: 16
Article B: es:Islas_Malvinas_(territorio_británico_de_ultramar)
Sections B: 12

BERTScore model: bert-base-multilingual-cased
Scoring every possible section pair...
Calculating BERTScore for 99 section pairs...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5876.44it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


100%|██████████| 2/2 [00:01<00:00,  1.06it/s]


computing greedy matching.


100%|██████████| 7/7 [00:00<00:00, 79.27it/s]


done in 1.97 seconds, 50.21 sentences/sec

Finding one-to-one alignment (threshold=0.70)...

Aligned pairs: 2
Total possible pairs: 192
Unmatched A sections: 14
Unmatched B sections: 10

SECTION ALIGNMENTS

en:Q9648:1  <->  es:Q9648:1
BERTScore F1: 0.7158

en:Q9648:7  <->  es:Q9648:6
BERTScore F1: 0.7806

COMPLETE BERTSCORE MATRIX
A section                     es:Q9648:0     es:Q9648:1     es:Q9648:2     es:Q9648:3     es:Q9648:4     es:Q9648:5     es:Q9648:6     es:Q9648:7     es:Q9648:8     es:Q9648:9     es:Q9648:10    es:Q9648:11    
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
en:Q9648:0                    0.6269         0.6093         0.6428         0.6292         0.6460         0.6052         0.6598         0.6502         0.0000         0.0000         0.4876         0.0000         
en:Q9648:1                    0.64

In [ ]:

propositions_list_es = []

for section in data_es["sections"]:
    propositions = extract_propositions(
            section=section,
            model=model_AP,
            tokenizer=tokenizer_AP,
            chunk_max_tokens=200,
            max_new_tokens=2048)

        # extend instead of append so that the final JSON
        # contains one flat list of propositions.

    propositions_list_es.extend(propositions)

    # -----------------------------------------------------------------
    # Save JSON
    # -----------------------------------------------------------------

with open("propositions_es.json", "w", encoding="utf-8") as file:
    json.dump( propositions_list_es, file, indent=4, ensure_ascii=False)

print("-----------------------------------")
print("Atomization completed")
print(f"Total propositions: {len(propositions_list_es)}")
print(f"Saved to: propositions_es.json")
print("-----------------------------------")


propositions_list_en = []

for section in data_en["sections"]:
    propositions = extract_propositions(
            section=section,
            model=model_AP,
            tokenizer=tokenizer_AP,
            chunk_max_tokens=200,
            max_new_tokens=2048)

        # extend instead of append so that the final JSON
        # contains one flat list of propositions.

    propositions_list_en.extend(propositions)

    # -----------------------------------------------------------------
    # Save JSON
    # -----------------------------------------------------------------

with open("propositions_en.json", "w", encoding="utf-8") as file:
    json.dump( propositions_list_en, file, indent=4, ensure_ascii=False)

print("-----------------------------------")
print("Atomization completed")
print(f"Total propositions: {len(propositions_list_en)}")
print(f"Saved to: propositions_en.json")
print("-----------------------------------")



triplets_en = extract_triplets( propositions_list_en, model_triplets, lenguage="en")

with open( "triplets_en.json", "w", encoding="utf-8" ) as f:
    json.dump(triplets_en, f, indent=4, ensure_ascii=False)
    print("Triples saved to triplets_en.json")

triplets_es = extract_triplets( propositions_list_es, model_triplets, lenguage="es")

with open( "triplets_es.json", "w", encoding="utf-8" ) as f:
    json.dump(triplets_es, f, indent=4, ensure_ascii=False)
    print("Triples saved to triplets_es.json")

In [59]:
with open("triplets_en.json", "r", encoding="utf-8") as file:
    triplets_en = json.load(file)
with open("triplets_es.json", "r", encoding="utf-8") as file:
    triplets_es = json.load(file)

list_results_omission = []
for value in section_pairs:
   pair_id = value['section_pair_id']
   section_id_en = value['section_id_a']
   section_id_es = value['section_id_b']

   triplets_en_section=[]
   for triplet in triplets_en:
      if section_id_en in triplet['triplet_id']:
         triplets_en_section.append(triplet)

   triplets_es_section=[]
   for triplet in triplets_es:
         if section_id_es in triplet['triplet_id']:
            triplets_es_section.append(triplet)

   print(f"Section ID EN: {section_id_en}, Section ID ES: {section_id_es} BERT Score: {value['alignment_score']}")
   print(f"Triplets in section {section_id_en}: {len(triplets_en_section)}")
   print(f"Triplets in section {section_id_es}: {len(triplets_es_section)}")
   print(f'Pair id: {pair_id}')

   metrics = calculate_omission(triplets_es_section, triplets_en_section)
   dict_metrics = {
      'section_id_en': section_id_en,
      'section_id_es': section_id_es,
      'pair_id': pair_id,
      'omission_metrics': metrics}

   list_results_omission.append(dict_metrics)



   
with open( "omission_metrics.json", "w", encoding="utf-8" ) as f:
    json.dump(list_results_omission, f, indent=4, ensure_ascii=False)
    print("Omission metrics saved to omission_metrics.json")





Section ID EN: en:Q9648:1, Section ID ES: es:Q9648:1 BERT Score: 0.7157663702964783
Triplets in section en:Q9648:1: 21
Triplets in section es:Q9648:1: 0
Pair id: en:Q9648:1::es:Q9648:1
Section ID EN: en:Q9648:7, Section ID ES: es:Q9648:6 BERT Score: 0.7805661559104919
Triplets in section en:Q9648:7: 6
Triplets in section es:Q9648:6: 11
Pair id: en:Q9648:7::es:Q9648:6
Omission metrics saved to omission_metrics.json


In [66]:
# manual
list_manual_section_pairs = [
    {
        "section_id_a": "es:Q9648:0",
        "section_id_b": "en:Q9648:0",
        "pair_id": "es:Q9648:0+en:Q9648:0+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:1",
        "section_id_b": "en:Q9648:1",
        "pair_id": "es:Q9648:1+en:Q9648:1+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:2",
        "section_id_b": "en:Q9648:2",
        "pair_id": "es:Q9648:2+en:Q9648:2+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:3",
        "section_id_b": "en:Q9648:3",
        "pair_id": "es:Q9648:3+en:Q9648:3+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:5",
        "section_id_b": "en:Q9648:6",
        "pair_id": "es:Q9648:5+en:Q9648:6+Manual",
        "alignment_score": "Manual"
    },
    {
        "section_id_a": "es:Q9648:6",
        "section_id_b": "en:Q9648:7",
        "pair_id": "es:Q9648:6+en:Q9648:7+Manual",
        "alignment_score": "Manual"
    }
]

In [67]:
with open("triplets_en.json", "r", encoding="utf-8") as file:
    triplets_en = json.load(file)
with open("triplets_es.json", "r", encoding="utf-8") as file:
    triplets_es = json.load(file)

list_results_omission_manual = []
for value in list_manual_section_pairs:
   pair_id = value['pair_id']
   section_id_en = value['section_id_b']
   section_id_es = value['section_id_a']

   triplets_en_section=[]
   for triplet in triplets_en:
      if section_id_en in triplet['triplet_id']:
         triplets_en_section.append(triplet)

   triplets_es_section=[]
   for triplet in triplets_es:
         if section_id_es in triplet['triplet_id']:
            triplets_es_section.append(triplet)

   print(f"Section ID EN: {section_id_en}, Section ID ES: {section_id_es} BERT Score: {value['alignment_score']}")
   print(f"Triplets in section {section_id_en}: {len(triplets_en_section)}")
   print(f"Triplets in section {section_id_es}: {len(triplets_es_section)}")
   print(f'Pair id: {pair_id}')

   metrics = calculate_omission(triplets_es_section, triplets_en_section)
   dict_metrics = {
      'section_id_en': section_id_en,
      'section_id_es': section_id_es,
      'pair_id': pair_id,
      'omission_metrics': metrics}

   list_results_omission_manual.append(dict_metrics)



   
with open( "omission_metrics_manual.json", "w", encoding="utf-8" ) as f:
    json.dump(list_results_omission_manual, f, indent=4, ensure_ascii=False)
    print("Omission metrics saved to omission_metrics_manual.json")



Section ID EN: en:Q9648:0, Section ID ES: es:Q9648:0 BERT Score: Manual
Triplets in section en:Q9648:0: 23
Triplets in section es:Q9648:0: 17
Pair id: es:Q9648:0+en:Q9648:0+Manual
Section ID EN: en:Q9648:1, Section ID ES: es:Q9648:1 BERT Score: Manual
Triplets in section en:Q9648:1: 21
Triplets in section es:Q9648:1: 0
Pair id: es:Q9648:1+en:Q9648:1+Manual
Section ID EN: en:Q9648:2, Section ID ES: es:Q9648:2 BERT Score: Manual
Triplets in section en:Q9648:2: 99
Triplets in section es:Q9648:2: 38
Pair id: es:Q9648:2+en:Q9648:2+Manual
Section ID EN: en:Q9648:3, Section ID ES: es:Q9648:3 BERT Score: Manual
Triplets in section en:Q9648:3: 57
Triplets in section es:Q9648:3: 29
Pair id: es:Q9648:3+en:Q9648:3+Manual
Section ID EN: en:Q9648:6, Section ID ES: es:Q9648:5 BERT Score: Manual
Triplets in section en:Q9648:6: 22
Triplets in section es:Q9648:5: 0
Pair id: es:Q9648:5+en:Q9648:6+Manual
Section ID EN: en:Q9648:7, Section ID ES: es:Q9648:6 BERT Score: Manual
Triplets in section en:Q9648:7